# Optuna tuning — 11 deep forecasters on Google Colab

`seq_len = 96`, `pred_len = 1`, **50 trials per model**, final run with **`--itr 5`**.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Run the cells in order. Steps 2–4 are setup; step 6 is the search and is the
long one. Everything is written to Google Drive, and studies **resume** — if
Colab disconnects, re-run the setup cells and then step 6 again, and it
carries on from the trial it reached instead of starting over.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'No GPU — switch Runtime > Change runtime type > T4 GPU (the search will be very slow on CPU)'

## 2. Mount Drive

Results and the Optuna database go here so they survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ProjectC_tuning'   # <- change if you like
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('results ->', DRIVE_DIR)

## 3. Clone the repository

`Mr0022/ProjectC` is **private**, so this needs a GitHub token with `repo`
scope: github.com → Settings → Developer settings → Personal access tokens.
The prompt is hidden, and the token is removed from the git remote right after
the clone so it is not left on disk.

In [ ]:
from getpass import getpass
import os, subprocess

BRANCH = 'claude/optuna-hyperparameter-tuning-11-models-hc0tgt'
REPO_DIR = '/content/ProjectC'

if not os.path.isdir(REPO_DIR):
    token = getpass('GitHub token (repo scope): ').strip()
    url = f'https://{token}@github.com/Mr0022/ProjectC.git'
    subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)
    # drop the token from .git/config
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin',
                    'https://github.com/Mr0022/ProjectC.git'], check=True)
    del token, url
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)

os.chdir(REPO_DIR)   # the code resolves models/ and data/ relatively
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

## 4. Install the dependencies Colab is missing

Colab already ships torch, numpy, pandas, scikit-learn and matplotlib. These
are the extras this repo needs — `ptwt` for WFTNet's wavelet transform,
`fast_pytorch_kmeans` for AdaWaveNet, `reformer-pytorch`/`local-attention`
because `layers/SelfAttention_Family.py` imports them at module load.

In [ ]:
!pip install -q optuna einops ptwt PyWavelets fast_pytorch_kmeans reformer-pytorch local-attention psutil

import torch, optuna
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| optuna', optuna.__version__)

## 5. Sanity check (~1 minute)

Two trials of the cheapest model. If this prints a best validation loss, the
environment is wired up correctly.

In [ ]:
!python tuning/optuna_tune.py --model DLinear --n_trials 2 --train_epochs 3 \
    --out_dir /content/_smoketest --checkpoint_dir /content/_ckpt 2>&1 | tail -5

## 6. The search — 50 trials per model

One model per call, so progress is saved after each. Editing `MODELS` lets you
run a subset (e.g. start with the cheap ones) or resume after a disconnect —
finished models return immediately, since the study is already in the database.

Rough cost on a T4: DLinear/FITS a few minutes each, PatchTST/TSLANet/
iTransformer/ModernTCN/AdaWaveNet/TimeMixer tens of minutes, MSGNet/TimesNet/
WFTNet up to a couple of hours. Budget for a long session, or run it in two.

Checkpoints go to local disk (`/content/_ckpt`), not Drive — they are
rewritten every improving epoch and deleted after each trial, so putting them
on a network mount would dominate the runtime.

In [ ]:
MODELS = ['DLinear', 'FITS', 'TSLANet', 'PatchTST', 'iTransformer', 'ModernTCN',
          'AdaWaveNet', 'TimeMixer', 'MSGNet', 'TimesNet', 'WFTNet']
N_TRIALS = 50

import subprocess, time
for model in MODELS:
    print(f'\n{"="*72}\n  {model}\n{"="*72}', flush=True)
    started = time.time()
    subprocess.run(['python', 'tuning/optuna_tune.py',
                    '--model', model,
                    '--n_trials', str(N_TRIALS),
                    '--out_dir', DRIVE_DIR,
                    '--checkpoint_dir', '/content/_ckpt'])
    print(f'{model} finished in {(time.time()-started)/60:.1f} min', flush=True)

### Optional: less noisy rankings

519 validation windows is a small sample, so differences under ~1 % are seed
noise. `--n_seeds 3` averages each configuration over three seeds before
ranking it — three times the cost, a much steadier winner. Use it if the
ranking is going into a paper.

In [ ]:
# for model in MODELS:
#     subprocess.run(['python', 'tuning/optuna_tune.py', '--model', model,
#                     '--n_trials', str(N_TRIALS), '--n_seeds', '3',
#                     '--out_dir', DRIVE_DIR, '--checkpoint_dir', '/content/_ckpt'])

## 7. Final runs — best config, `--itr 5`

Re-trains each winner through `run.py` with **five seeds (2021–2025)** and
prints mean ± std, min and max for every HAR-comparable metric — MSE/MAE on
the `ln(RV)` scale, QLIKE and MSE_RV/MAE_RV back on the variance scale, all
directly comparable with `python HAR-RV_RUN.PY --log`.

In [ ]:
import json, glob, subprocess, os

for path in sorted(glob.glob(os.path.join(DRIVE_DIR, '*_best.json'))):
    best = json.load(open(path))
    argv = best['command'].split()[3:] + ['--des', 'best', '--itr', '5']
    print(f'\n{"="*72}\n  {best["model"]}  (best val loss {best["best_val_loss"]:.6f})\n{"="*72}', flush=True)
    subprocess.run(['python', '-u', 'run.py'] + argv)

## 8. Summary table

Best configuration per model, ranked by validation loss.

In [ ]:
import json, glob, os
import pandas as pd

rows = []
for path in sorted(glob.glob(os.path.join(DRIVE_DIR, '*_best.json'))):
    b = json.load(open(path))
    rows.append({'model': b['model'],
                 'val_loss': round(b['best_val_loss'], 6),
                 'trials': b['n_complete'],
                 'minutes': round(b['seconds'] / 60, 1),
                 **b['best_params']})

df = pd.DataFrame(rows).sort_values('val_loss').reset_index(drop=True)
df.to_csv(os.path.join(DRIVE_DIR, 'best_configs.csv'), index=False)
display(df[['model', 'val_loss', 'trials', 'minutes']])
df

In [ ]:
# the exact command line that reproduces each winner
for path in sorted(glob.glob(os.path.join(DRIVE_DIR, '*_best.json'))):
    b = json.load(open(path))
    print(f'# {b["model"]}  (val {b["best_val_loss"]:.6f})\n{b["command"]}\n')

## Notes

* **Resuming** — studies live in `optuna.db` on Drive. Re-running step 6 after
  a disconnect continues from where it stopped; a model that already has its
  50 trials returns at once.
* **What is being minimised** — the lowest validation loss reached during
  training, i.e. the epoch `EarlyStopping` checkpoints. The test split is
  never touched during a search; it appears only in step 7.
* **Equal protocol** — every model gets the same 50 trials and the same
  learning-rate/batch/schedule ranges, so the table compares architectures,
  not tuning effort.
* **Search spaces** — `tuning/README.md` documents every range and every
  architectural constraint; `tuning/search_spaces.py` is the source of truth.